# Pi0.5 STC：夹爪开合对比潜方向

这个 notebook 不重训 transcoder，也不改 Jayden 的 rollout notebook。它读取 LIBERO 示教，在 probe 模式下收集已训练 STC 的稀疏激活 `z`，用 open 减 close 得到一个 `(layer, tau)` 上的方向，再通过现有的 probe 残差做 `z + alpha * v`。

夹爪是动作的第 7 维（下标 6，`grip`）。open/close 的符号不写死：先看 `action[:, 6]` 的两个峰，再用 `observation.state` 最后两维的指缝决定哪一峰是张开。

Colab 上请先跑过 Jayden notebook 的安装单元，这样 `/content/lerobot-venv` 和 Drive 上的 STC checkpoint 已经就绪。本实验不启动 LIBERO 仿真器。`REPO_BRANCH` 必须包含 `scripts/run_gripper_contrastive_steer.py`。

In [ ]:
# @title Controls

DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/jayden-transcoder-feature-inspection"  # @param {type:"string"}
POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_10"]
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}
TRANSCODER_LOCAL_PATH = ""  # @param {type:"string"}
HORIZON = 5  # @param {type:"integer"}
TRAIN_FRACTION = 0.7  # @param {type:"number"}
MAX_PER_GROUP = 2  # @param {type:"integer"}
MAX_HOLDOUT_PER_LABEL = 8  # @param {type:"integer"}
TOP_K = 32  # @param {type:"integer"}
MIN_CONSISTENCY = 0.7  # @param {type:"number"}
ALPHAS = "0,0.5,1,2,4"  # @param {type:"string"}
SEED = 0  # @param {type:"integer"}
NUM_INFERENCE_STEPS = 10  # @param {type:"integer"}

print("suite", SUITE)
print("alphas", ALPHAS)


In [ ]:
# @title Resolve repo, checkpoint, and the existing LeRobot runtime

from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOCAL_REPO = Path("/content/groot-run")
    if LOCAL_REPO.exists():
        subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
    PYTHON = Path("/content/lerobot-venv/bin/python")
    if not PYTHON.exists():
        raise RuntimeError(
            "Missing /content/lerobot-venv. Run the install cell in "
            "notebooks/pi05_libero_transcoder_colab_jayden.ipynb first."
        )
    token_file = Path(DRIVE_ROOT) / "secrets/HF_TOKEN.txt"
    if token_file.exists():
        os.environ["HF_TOKEN"] = token_file.read_text(encoding="utf-8").strip()
        os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    checkpoint = Path(TRANSCODER_LOCAL_PATH) if TRANSCODER_LOCAL_PATH else Path(DRIVE_ROOT) / TRANSCODER_DRIVE_PATH
else:
    LOCAL_REPO = Path.cwd()
    if not (LOCAL_REPO / "scripts" / "run_gripper_contrastive_steer.py").exists():
        LOCAL_REPO = Path("/content/groot-run")
    PYTHON = Path(sys.executable)
    checkpoint = Path(TRANSCODER_LOCAL_PATH) if TRANSCODER_LOCAL_PATH else LOCAL_REPO / TRANSCODER_DRIVE_PATH

script = LOCAL_REPO / "scripts" / "run_gripper_contrastive_steer.py"
if not script.exists():
    raise FileNotFoundError(f"Experiment script is not on this branch: {script}")
if not checkpoint.exists():
    raise FileNotFoundError(f"STC checkpoint not found: {checkpoint}")

OUTPUT_DIR = LOCAL_REPO / "outputs" / "gripper_contrastive"
print("repo", LOCAL_REPO)
print("python", PYTHON)
print("checkpoint", checkpoint)
print("output", OUTPUT_DIR)


In [ ]:
# @title Run contrastive identification and held-out steering

cmd = [
    str(PYTHON),
    "-u",
    str(script),
    "--policy-path", POLICY_PATH,
    "--checkpoint", str(checkpoint),
    "--suite", SUITE,
    "--output-dir", str(OUTPUT_DIR),
    "--horizon", str(HORIZON),
    "--train-fraction", str(TRAIN_FRACTION),
    "--max-per-group", str(MAX_PER_GROUP),
    "--max-holdout-per-label", str(MAX_HOLDOUT_PER_LABEL),
    "--top-k", str(TOP_K),
    "--min-consistency", str(MIN_CONSISTENCY),
    "--alphas", ALPHAS,
    "--seed", str(SEED),
    "--num-inference-steps", str(NUM_INFERENCE_STEPS),
]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, cwd=LOCAL_REPO, check=True)


In [ ]:
# @title Show selection, top features, and steering plots

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

report = json.loads((OUTPUT_DIR / "selection_report.json").read_text(encoding="utf-8"))
convention = report["convention"]
agreement = report.get("probe_agreement", {})
side = "高 action 是 open" if convention["open_is_high"] else "高 action 是 close"
display(Markdown(
    "\n".join([
        f"- 扫描帧数：{report['n_frames']}",
        f"- 稳定 open / close：{report['n_stable_open']} / {report['n_stable_close']}",
        f"- 配对数：{report['n_pairs']}",
        f"- probe 用的 train / holdout：{report['n_train_probed']} / {report['n_holdout_probed']}",
        f"- 夹爪下标 6，符号：{side}",
        f"- 阈值：{convention['low_threshold']:.4f} , {convention['high_threshold']:.4f}",
        f"- 指缝相关：{convention['finger_gap_correlation']:.4f}",
        f"- 选中的 STC：layer {report['selected_layer']}，tau {report['selected_tau']}",
        f"- 示教标签和 baseline 预测一致：{agreement.get('n_agree', 0)} / {agreement.get('n', 0)}",
    ])
))

ranking = pd.read_csv(OUTPUT_DIR / "feature_ranking.csv")
display(ranking.head(20))
steering = pd.read_csv(OUTPUT_DIR / "steering_results.csv")
summary = (
    steering.groupby(["direction_kind", "control", "alpha"], as_index=False)[["delta_gripper", "other_dims_mean_abs"]]
    .mean()
    .sort_values(["direction_kind", "control", "alpha"])
)
display(summary)

for name in [
    "gripper_action_histogram.png",
    "top_feature_deltas.png",
    "gripper_effect_vs_alpha.png",
    "gripper_vs_random.png",
    "action_dimension_effects.png",
]:
    path = OUTPUT_DIR / name
    display(Markdown(f"### {name}"))
    display(Image(filename=str(path)))

if IN_COLAB:
    drive_out = Path(DRIVE_ROOT) / "outputs" / "gripper_contrastive"
    drive_out.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            target = drive_out / path.name
            target.write_bytes(path.read_bytes())
    print("copied results to", drive_out)
